# RAG System for QMUL Policy Search
## 1. Document preprocessing

This cleaned notebook keeps the original preprocessing workflow but makes it easier to run and maintain:

1. Configure input and output paths in one place.
2. Ingest PDFs with a PyMuPDF-first strategy and a `pdfplumber` fallback.
3. Clean page text by removing page numbers, repeated headers and footers, and whitespace artifacts.
4. Build section-level chunks for indexing.
5. Export page-level, section-level, and chunk-level outputs.

- **Corpus:** 20 QMUL Finance & Governance policy documents → 237 chunks  
- **Test collection:** 40 queries (2 per document) · 1 broad query and 1 narrower sub-section query
- **Stage 1 — Classical IR:** BM25, VSM, RRF evaluated with P@10, R@10, MRR@10, nDCG@10  
- **Stage 2 — RAG Generation:** LLM answers scored for Faithfulness, Answer Relevance, Context Utilization  

**Improvements over baseline:**
- Section-title boosting in BM25 and VSM indexes (Zhai & Lafferty, 2001)
- BM25 parameter tuning: k1=1.2, b=0.5 (better for short policy chunks)
- Faithfulness — LLM decomposes answer into atomic claims, judges each against the context
- ResponseRelevancy — LLM generates reverse questions from the answer, checks embedding similarity
- ContextUtilization — LLM judges how well the retrieved context was actually used
### 1.1: Set-up

In [ ]:
%pip install -q pymupdf pdfplumber tqdm pandas pyarrow

<frozen posixpath>:82: RuntimeWarning: coroutine 'ClientResponse.json' was never awaited


 ### 1.2: Link Drive
 Store all the documents which should be in a folder named 'PDFs' in drive under a folder named 'IR_Qmary'.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

AUTO_MOUNT_GOOGLE_DRIVE = True

if IN_COLAB and AUTO_MOUNT_GOOGLE_DRIVE:
    drive.mount("/content/drive")

DEFAULT_COLAB_ROOT = Path("/content/drive/MyDrive/IR_Qmary")
DEFAULT_LOCAL_ROOT = Path.cwd()

WORK_ROOT = DEFAULT_COLAB_ROOT if IN_COLAB else DEFAULT_LOCAL_ROOT
PDF_DIR = WORK_ROOT / "PDFs"
OUTPUT_DIR = WORK_ROOT

OUT_PATH = OUTPUT_DIR / "ingested_pages.csv"
SECTIONS_OUT = OUTPUT_DIR / "sections.csv"
CHUNK_OUT = OUTPUT_DIR / "clean_chunks.csv"
AUDIT_DIR = OUTPUT_DIR / "chunk_audit"

pdf_paths = sorted(str(path) for path in PDF_DIR.glob("*.pdf"))

print(f"Running in Colab: {IN_COLAB}")
print(f"PDF_DIR: {PDF_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"PDF count: {len(pdf_paths)}")

if not pdf_paths:
    raise FileNotFoundError(
        f"No PDF files found in {PDF_DIR}. Update PDF_DIR before running the ingestion cells."
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab: True
PDF_DIR: /content/drive/MyDrive/IR_Qmary/PDFs
OUTPUT_DIR: /content/drive/MyDrive/IR_Qmary
PDF count: 20


### 1.3: Extraction and Ingestion

In [ ]:
import fitz  # PyMuPDF
from tqdm.auto import tqdm

def extract_pdf_pages_pymupdf(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    records = []
    doc_id = os.path.splitext(os.path.basename(pdf_path))[0]

    for i in range(len(doc)):
        page = doc[i]
        text = page.get_text("text") or ""
        records.append({
            "doc_id": doc_id,
            "source_path": pdf_path,
            "file_name": os.path.basename(pdf_path),
            "page_num": i + 1,
            "text": text
        })

    doc.close()
    return records

In [ ]:
import re
import pdfplumber

def extract_pdf_pages_pdfplumber(pdf_path: str) -> list[dict]:
    records = []
    doc_id = os.path.splitext(os.path.basename(pdf_path))[0]

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text() or ""
            records.append({
                "doc_id": doc_id,
                "source_path": pdf_path,
                "file_name": os.path.basename(pdf_path),
                "page_num": i + 1,
                "text": text
            })
    return records

def is_probably_scanned(page_text: str, min_chars: int = 80, min_alpha_ratio: float = 0.35) -> bool:
    if not isinstance(page_text, str):
        return True
    compact = re.sub(r"\s+", "", page_text)
    if len(compact) < min_chars:
        return True
    alpha_ratio = sum(ch.isalpha() for ch in compact) / max(1, len(compact))
    return alpha_ratio < min_alpha_ratio

In [ ]:
def ingest_pdf(pdf_path: str) -> list[dict]:
    recs = extract_pdf_pages_pymupdf(pdf_path)

    empty_ratio = sum(is_probably_scanned(r["text"]) for r in recs) / max(1, len(recs))
    if empty_ratio > 0.6:
        recs = extract_pdf_pages_pdfplumber(pdf_path)

    for r in recs:
        r["likely_scanned"] = is_probably_scanned(r["text"])
    return recs

In [ ]:
import pandas as pd

all_records = []
for path in tqdm(pdf_paths, desc="Ingesting PDFs"):
    all_records.extend(ingest_pdf(path))

df_pages = pd.DataFrame(all_records)
df_pages.head()

Ingesting PDFs:   0%|          | 0/20 [00:00<?, ?it/s]

,doc_id,source_path,file_name,page_num,text,likely_scanned
0,Anti-Bribery-and-Corruption-Policy-updated-Jun...,/content/drive/MyDrive/IR_Qmary/PDFs/Anti-Brib...,Anti-Bribery-and-Corruption-Policy-updated-Jun...,1,\n \n \n \n \n \n \n \n \n \n \n \n \n \n \n ...,False
1,Anti-Bribery-and-Corruption-Policy-updated-Jun...,/content/drive/MyDrive/IR_Qmary/PDFs/Anti-Brib...,Anti-Bribery-and-Corruption-Policy-updated-Jun...,2,\n \n- 2 - \nContents \n \n \n1 \nIntroductio...,True
2,Anti-Bribery-and-Corruption-Policy-updated-Jun...,/content/drive/MyDrive/IR_Qmary/PDFs/Anti-Brib...,Anti-Bribery-and-Corruption-Policy-updated-Jun...,3,\n \n- 3 - \n1 \nIntroduction and Policy Stat...,False
3,Anti-Bribery-and-Corruption-Policy-updated-Jun...,/content/drive/MyDrive/IR_Qmary/PDFs/Anti-Brib...,Anti-Bribery-and-Corruption-Policy-updated-Jun...,4,\n \n- 4 - \n2 \nWhat is Bribery and Corrupti...,False
4,Anti-Bribery-and-Corruption-Policy-updated-Jun...,/content/drive/MyDrive/IR_Qmary/PDFs/Anti-Brib...,Anti-Bribery-and-Corruption-Policy-updated-Jun...,5,\n \n- 5 - \n \nBribing a foreign official \n...,False


In [ ]:
print("Total pages:", len(df_pages))
print("Unique docs:", df_pages["doc_id"].nunique())
print("Likely scanned pages:", df_pages["likely_scanned"].sum())
print("Empty pages:", (df_pages["text"].str.strip() == "").sum())

Total pages: 252
Unique docs: 20
Likely scanned pages: 15
Empty pages: 0


In [ ]:
df_pages.to_parquet(OUT_PATH, index=False)
print("Saved page-level data to:", OUT_PATH)


Saved page-level data to: /content/drive/MyDrive/IR_Qmary/ingested_pages.csv


## 2. Clean Extracted Text

This stage removes page-number artifacts, repeated headers or footers, hyphenation splits, and excess whitespace.

Justification: Page numbers sometimes break the regex header detection for chunking, and do not add semantic value. Repeated headers or footers add no semantic value. Hyphenation splits and excess whitespace are removed for more accurate representation.

### 2.1: Page number removal


In [ ]:
import re
import unicodedata
from collections import Counter
import pandas as pd

PAGE_LABEL_RE = re.compile(r"^\s*page\s+\d+(\s+of\s+\d+)?\s*$", re.I)
PAGE_OF_RE    = re.compile(r"^\s*\d+\s+of\s+\d+\s*$", re.I)
BARE_NUM_RE   = re.compile(r"^\s*\d{1,4}\s*$")

def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", text)

def fix_hyphenation(text: str) -> str:
    return re.sub(r"(\w)-\n(\w)", r"\1\2", text)

def normalize_newlines_keep_lines(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = "\n".join(ln.strip() for ln in text.split("\n"))
    return text.strip()

def collapse_whitespace_keep_newlines(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()

def is_obvious_page_number_line(line: str) -> bool:
    if not isinstance(line, str):
        return False
    s = line.strip()
    if not s:
        return False
    return (
        PAGE_LABEL_RE.match(s) is not None
        or PAGE_OF_RE.match(s) is not None
    )

def is_bare_number_line(line: str) -> bool:
    if not isinstance(line, str):
        return False
    s = line.strip()
    if not s:
        return False
    return BARE_NUM_RE.match(s) is not None

def remove_boundary_page_number_lines(text: str, boundary_window: int = 3) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    lines = text.replace("\r\n", "\n").replace("\r", "\n").split("\n")
    nonblank_idx = [i for i, ln in enumerate(lines) if ln.strip()]
    if not nonblank_idx:
        return ""

    first_idxs = set(nonblank_idx[:boundary_window])
    last_idxs = set(nonblank_idx[-boundary_window:])
    boundary_idxs = first_idxs | last_idxs

    kept = []
    for i, ln in enumerate(lines):
        s = ln.strip()

        if not s:
            kept.append(ln)
            continue

        if is_obvious_page_number_line(s):
            continue

        if i in boundary_idxs and is_bare_number_line(s):
            continue

        kept.append(ln)

    return "\n".join(kept).strip()

def basic_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = normalize_unicode(text)
    text = fix_hyphenation(text)
    text = normalize_newlines_keep_lines(text)
    text = remove_boundary_page_number_lines(text, boundary_window=3)
    text = collapse_whitespace_keep_newlines(text)
    return text

### 2.2: Repeated header and footer removal

In [ ]:
def get_repeated_lines_for_doc(doc_df: pd.DataFrame, min_len=12, freq_ratio=0.4):
    pages = doc_df.sort_values("page_num")["text"].fillna("").tolist()
    n_pages = max(1, len(pages))

    line_counts = Counter()

    for page_text in pages:
        page_text = normalize_unicode(page_text)
        page_text = normalize_newlines_keep_lines(page_text)
        page_text = remove_boundary_page_number_lines(page_text, boundary_window=3)

        raw_lines = [ln.strip() for ln in page_text.split("\n")]
        raw_lines = [ln for ln in raw_lines if ln and len(ln) >= min_len]

        for ln in set(raw_lines):
            line_counts[ln] += 1

    threshold = max(2, int(freq_ratio * n_pages))
    repeated = {ln for ln, c in line_counts.items() if c >= threshold}
    return repeated

def remove_lines(text: str, banned_lines: set[str]) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    text = normalize_unicode(text)
    text = normalize_newlines_keep_lines(text)

    lines = text.split("\n")
    nonblank_idx = [i for i, ln in enumerate(lines) if ln.strip()]
    first_idxs = set(nonblank_idx[:3])
    last_idxs = set(nonblank_idx[-3:])
    boundary_idxs = first_idxs | last_idxs

    kept = []
    for i, ln in enumerate(lines):
        s = ln.strip()

        if not s:
            kept.append(ln)
            continue

        if is_obvious_page_number_line(s):
            continue

        if i in boundary_idxs and is_bare_number_line(s):
            continue

        if s in banned_lines:
            continue

        kept.append(ln)

    return "\n".join(kept).strip()

In [ ]:
df_pages = df_pages.copy()

banned_by_doc = {}
for doc_id, g in df_pages.groupby("doc_id"):
    banned_by_doc[doc_id] = get_repeated_lines_for_doc(
        g,
        min_len=12,
        freq_ratio=0.4,
    )

df_pages["text_no_boilerplate"] = df_pages.apply(
    lambda row: remove_lines(row["text"], banned_by_doc.get(row["doc_id"], set())),
    axis=1,
)

df_pages["clean_text"] = df_pages["text_no_boilerplate"].apply(basic_clean)


## 3. Build Section-Level Chunks

This version keeps one chunk per detected top-level section and falls back to a full-document chunk when headings are unreliable.

First, the code normalises each page by cleaning whitespace, removing hidden characters, and standardising heading formats. It then detects repeated lines across pages so that likely headers, footers, and page numbers can be removed.

Next, the pipeline identifies top-level section headings, such as numbered headings or appendix titles, and groups all text belonging to each section across page boundaries into a single section block. This means one detected top-level section becomes one chunk, which helps preserve semantic coherence for retrieval. If no reliable headings are found, the code uses a fallback strategy to detect paragraph-style headings, and if that also fails, it stores the whole document as one FULL_DOCUMENT chunk.

Finally, the output is converted into chunk-level and section-level dataframes with metadata such as document ID, section title, page range, word count, and a unique chunk ID. In a RAG workflow, these chunks can then be indexed for retrieval, allowing the model to return answers grounded in complete document sections rather than fragmented text.

In [ ]:
import re
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd

DEBUG_DOC_ID = None
DEBUG_PAGE_NUM = 1
DEBUG_ON = False

_ZW = re.compile(r"[\u200B-\u200D\uFEFF]")
_WS = re.compile(r"\s+")

def _clean_line(line: str) -> str:
    if not isinstance(line, str):
        return ""
    line = line.replace("\u00A0", " ")
    line = _ZW.sub("", line)
    line = line.strip()
    line = _WS.sub(" ", line)

    # normalize "1 ." -> "1."
    line = re.sub(r"^(\d+)\s+\.", r"\1.", line)

    # normalize "1.Intro" -> "1. Intro" but not "1.1"
    line = re.sub(r"^(\d+)\.([A-Za-z(\[])", r"\1. \2", line)
    return line

def normalize_lines(page_text: str) -> List[str]:
    raw = (page_text or "").split("\n")
    lines = [_clean_line(x) for x in raw]
    return [x for x in lines if x]

def normalize_lines_keep_blanks(page_text: str) -> List[Optional[str]]:
    raw = (page_text or "").split("\n")
    out: List[Optional[str]] = []
    for x in raw:
        cl = _clean_line(x)
        out.append(cl if cl else None)
    return out

MAJOR_HEADING_DOT = re.compile(r"^\d+\.\s*\S")           # "2. Policy"
MAJOR_HEADING_NODOT = re.compile(r"^\d{1,2}\s+[A-Z]")   # "8 Responsibilities" but not "2017 Act"
SUBCLAUSE_NUM = re.compile(r"^\d+\.\d+(\.\d+)*\b")      # "2.1", "1.2.3"
TOC_DOTS = re.compile(r"\.{3,}")
ENDS_WITH_PAGE = re.compile(r"\s\d+$")

BULLETISH = re.compile(r"^[•\-\u2022]")
APPENDIX = re.compile(r"^(Appendix|APPENDIX)\b", re.I)
YEAR_LIKE_START = re.compile(r"^(19|20)\d{2}(\.|\s|$)")

def looks_like_numeric_major_heading(line: str) -> bool:
    line = _clean_line(line)
    if not line:
        return False

    # reject year-like starts such as:
    # "1998. However..."
    # "2017 Act also includes..."
    # "2024"
    if YEAR_LIKE_START.match(line):
        return False

    if TOC_DOTS.search(line):
        return False

    if (MAJOR_HEADING_DOT.match(line) or MAJOR_HEADING_NODOT.match(line)) and ENDS_WITH_PAGE.search(line):
        return False

    if len(line) < 2 or len(line) > 160:
        return False

    if SUBCLAUSE_NUM.match(line):
        return False

    return bool(MAJOR_HEADING_DOT.match(line) or MAJOR_HEADING_NODOT.match(line))

def looks_like_appendix_heading(line: str) -> bool:
    line = _clean_line(line)
    if not line:
        return False
    if len(line) < 6 or len(line) > 180:
        return False
    return bool(APPENDIX.match(line))

In [ ]:
NUM_ONLY = re.compile(r"^\d+$")
DOT_ONLY = re.compile(r"^\.$")
NUM_DOT = re.compile(r"^\d+\.$")

def _maybe_join_split_numeric_heading(lines: List[str]) -> List[str]:
    out: List[str] = []
    i = 0

    def is_title_line(s: str) -> bool:
        s = _clean_line(s)
        if not s:
            return False
        if BULLETISH.match(s):
            return False
        if SUBCLAUSE_NUM.match(s):
            return False
        if re.fullmatch(r"\d+(\.\d+)*", s):
            return False
        if not re.match(r"^[A-Z(\[]", s):
            return False
        return True

    while i < len(lines):
        a = _clean_line(lines[i])

        # Case A: "1" "." "Title"
        if NUM_ONLY.match(a) and i + 2 < len(lines):
            b = _clean_line(lines[i + 1])
            c = _clean_line(lines[i + 2])
            if DOT_ONLY.match(b) and is_title_line(c):
                out.append(f"{a}. {c}")
                i += 3
                continue

        # Case B: "1." "Title"
        if NUM_DOT.match(a) and i + 1 < len(lines):
            b = _clean_line(lines[i + 1])
            if is_title_line(b):
                out.append(f"{a.rstrip('.')}. {b}")
                i += 2
                continue

        # Case C: "1" "Title"
        if NUM_ONLY.match(a) and i + 1 < len(lines):
            b = _clean_line(lines[i + 1])
            if is_title_line(b):
                out.append(f"{a}. {b}")
                i += 2
                continue

        out.append(a)
        i += 1

    return [x for x in out if x]


_INLINE_HEADING_DOT = re.compile(r"(?<=[\.\)\]])\s+(\d+\.\s+[A-Z])")
_INLINE_HEADING_NODOT = re.compile(r"(?<=[\.\)\]])\s+(\d{1,2}\s+[A-Z])")
_INLINE_HEADING_AFTER_WORD = re.compile(r"(?<=[a-z])\s+(\d+\.\s+[A-Z])")

def _insert_heading_breaks(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ""
    text = _INLINE_HEADING_DOT.sub(r"\n\1", text)
    text = _INLINE_HEADING_NODOT.sub(r"\n\1", text)
    text = _INLINE_HEADING_AFTER_WORD.sub(r"\n\1", text)
    return text


STANDALONE_NUM_TOKEN = re.compile(r"^\d+\.$")
STANDALONE_ALPHA_TOKEN = re.compile(r"^[a-zA-Z]\)$")

def compute_repeated_lines(doc_df: pd.DataFrame, min_frac: float = 0.4, max_len: int = 120) -> set:
    n_pages = doc_df["page_num"].nunique()
    if n_pages <= 1:
        return set()

    line_page_count = Counter()
    for _, row in doc_df.iterrows():
        for ln in set(normalize_lines(row.get("clean_text", "") or "")):
            if len(ln) > max_len:
                continue

            if looks_like_appendix_heading(ln) or looks_like_numeric_major_heading(ln):
                continue

            if STANDALONE_NUM_TOKEN.fullmatch(ln):
                continue
            if STANDALONE_ALPHA_TOKEN.fullmatch(ln):
                continue

            line_page_count[ln] += 1

    thresh = max(2, int(n_pages * min_frac))
    repeated = {ln for ln, c in line_page_count.items() if c >= thresh}

    repeated |= {
        ln for ln in repeated
        if re.fullmatch(r"Page\s*\d+(\s*of\s*\d+)?", ln, flags=re.I) is not None
    }
    return repeated

In [ ]:
def normalize_lines_without_headers(page_text: str, repeated_lines: set) -> List[str]:
    lines = normalize_lines(page_text)
    if not repeated_lines:
        return lines
    return [ln for ln in lines if ln not in repeated_lines]

def normalize_lines_keep_blanks_without_headers(page_text: str, repeated_lines: set) -> List[Optional[str]]:
    lines = normalize_lines_keep_blanks(page_text)
    if not repeated_lines:
        return lines
    out: List[Optional[str]] = []
    for ln in lines:
        if ln is None:
            out.append(None)
        else:
            out.append(None if ln in repeated_lines else ln)
    return out

def _is_fallback_heading(candidate: str, next_line: Optional[str]) -> bool:
    s = _clean_line(candidate)
    if not s:
        return False

    if re.match(r"^(19|20)\d{2}\.", s):
        return False

    if len(s) < 3 or len(s) > 110:
        return False
    if BULLETISH.match(s):
        return False
    if SUBCLAUSE_NUM.match(s) or looks_like_numeric_major_heading(s):
        return False
    if s.endswith("."):
        return False

    words = s.split()
    if len(words) >= 2:
        cap_words = sum(1 for w in words if w[:1].isupper())
        if cap_words / len(words) < 0.4:
            return False

    nxt = _clean_line(next_line) if isinstance(next_line, str) else ""
    if not nxt:
        return True
    if len(nxt) >= 40:
        return True
    if SUBCLAUSE_NUM.match(nxt) or BULLETISH.match(nxt):
        return True
    return True

In [ ]:
@dataclass
class SectionBuffer:
    doc_id: str
    section_title: str
    parts: List[Tuple[int, str]]  # (page_num, line)


def _full_document_block(doc_df: pd.DataFrame, title_fallback: str = "FULL_DOCUMENT") -> List[Dict[str, Any]]:
    doc_df = doc_df.sort_values("page_num").reset_index(drop=True)
    doc_id = str(doc_df["doc_id"].iloc[0])

    full_parts: List[Tuple[int, str]] = []
    for _, row in doc_df.iterrows():
        page_num = int(row["page_num"])
        page_text = (row.get("clean_text", "") or "").strip()
        if not page_text:
            continue
        lines = normalize_lines(page_text)
        for ln in lines:
            full_parts.append((page_num, ln))

    if not full_parts:
        return []

    pages = [p for p, _ in full_parts]
    section_text = "\n".join(t for _, t in full_parts).strip()

    return [{
        "doc_id": doc_id,
        "section_title": title_fallback,
        "page_start": min(pages),
        "page_end": max(pages),
        "section_text": section_text,
        "n_words": len(section_text.split()),
    }]


def _build_sections_numeric_plus_appendix(
    doc_df: pd.DataFrame,
    title_fallback: str,
    header_footer_min_frac: float
) -> Tuple[List[Dict[str, Any]], int]:

    doc_df = doc_df.sort_values("page_num").reset_index(drop=True)
    doc_id = str(doc_df["doc_id"].iloc[0])
    repeated = compute_repeated_lines(doc_df, min_frac=header_footer_min_frac)

    blocks: List[Dict[str, Any]] = []
    current = SectionBuffer(doc_id=doc_id, section_title=title_fallback, parts=[])
    major_count = 0
    in_appendix = False

    def flush(buf: SectionBuffer):
        if not buf.parts:
            return

        pages = [p for p, _ in buf.parts]
        section_text = "\n".join(t for _, t in buf.parts).strip()

        if not section_text:
            return

        blocks.append({
            "doc_id": buf.doc_id,
            "section_title": buf.section_title,
            "page_start": min(pages),
            "page_end": max(pages),
            "section_text": section_text,
            "n_words": len(section_text.split()),
        })

    for _, row in doc_df.iterrows():
        page_num = int(row["page_num"])
        page_text = _insert_heading_breaks((row.get("clean_text", "") or "")).strip()
        if not page_text:
            continue

        lines = normalize_lines_without_headers(page_text, repeated)

        if lines:
            if re.fullmatch(r"\d{1,4}", lines[0]):
                lines = lines[1:]
            if lines and re.fullmatch(r"\d{1,4}", lines[-1]):
                lines = lines[:-1]

        lines = _maybe_join_split_numeric_heading(lines)

        if DEBUG_ON and (DEBUG_DOC_ID is None or doc_id == DEBUG_DOC_ID) and page_num == DEBUG_PAGE_NUM:
            print(f"\n--- DEBUG doc={doc_id} page={page_num} ---")
            for ln in lines[:80]:
                print(repr(ln))

        for ln in lines:
            ln = _clean_line(ln)

            if looks_like_appendix_heading(ln):
                flush(current)
                current = SectionBuffer(doc_id=doc_id, section_title=ln, parts=[(page_num, ln)])
                in_appendix = True
                major_count += 1
                continue

            if (not in_appendix) and looks_like_numeric_major_heading(ln):
                flush(current)
                current = SectionBuffer(doc_id=doc_id, section_title=ln, parts=[(page_num, ln)])
                major_count += 1
            else:
                current.parts.append((page_num, ln))

    flush(current)
    return blocks, major_count

In [ ]:
def _build_sections_fallback_paragraph_headings(
    doc_df: pd.DataFrame,
    title_fallback: str,
    header_footer_min_frac: float
) -> List[Dict[str, Any]]:

    doc_df = doc_df.sort_values("page_num").reset_index(drop=True)
    doc_id = str(doc_df["doc_id"].iloc[0])
    repeated = compute_repeated_lines(doc_df, min_frac=header_footer_min_frac)

    blocks: List[Dict[str, Any]] = []
    current = SectionBuffer(doc_id=doc_id, section_title=title_fallback, parts=[])

    def flush(buf: SectionBuffer):
        if not buf.parts:
            return

        pages = [p for p, _ in buf.parts]
        section_text = "\n".join(t for _, t in buf.parts).strip()

        if not section_text:
            return

        blocks.append({
            "doc_id": buf.doc_id,
            "section_title": buf.section_title,
            "page_start": min(pages),
            "page_end": max(pages),
            "section_text": section_text,
            "n_words": len(section_text.split()),
        })

    for _, row in doc_df.iterrows():
        page_num = int(row["page_num"])
        page_text = (row.get("clean_text", "") or "").strip()
        if not page_text:
            continue

        lines = normalize_lines_keep_blanks_without_headers(page_text, repeated)

        nonblank = [i for i, x in enumerate(lines) if x is not None]
        if nonblank:
            first_i = nonblank[0]
            last_i = nonblank[-1]
            if isinstance(lines[first_i], str) and re.fullmatch(r"\d{1,4}", lines[first_i].strip()):
                lines[first_i] = None
            if isinstance(lines[last_i], str) and re.fullmatch(r"\d{1,4}", lines[last_i].strip()):
                lines[last_i] = None

        prev_blank = True

        def next_nonblank(idx: int) -> Optional[str]:
            j = idx + 1
            while j < len(lines):
                if lines[j] is not None:
                    return lines[j]
                j += 1
            return None

        for i, ln in enumerate(lines):
            if ln is None:
                prev_blank = True
                continue

            if prev_blank:
                nxt = next_nonblank(i)
                if _is_fallback_heading(ln, nxt):
                    h = _clean_line(ln)
                    flush(current)
                    current = SectionBuffer(doc_id=doc_id, section_title=h, parts=[(page_num, h)])
                    prev_blank = False
                    continue

            current.parts.append((page_num, _clean_line(ln)))
            prev_blank = False

    flush(current)
    return blocks


def build_sections_across_pages(
    doc_df: pd.DataFrame,
    title_fallback: str = "PREAMBLE",
    header_footer_min_frac: float = 0.4
) -> List[Dict[str, Any]]:
    doc_df = doc_df.sort_values("page_num").reset_index(drop=True)
    assert doc_df["doc_id"].nunique() == 1

    blocks, major_count = _build_sections_numeric_plus_appendix(
        doc_df,
        title_fallback=title_fallback,
        header_footer_min_frac=header_footer_min_frac
    )

    if major_count == 0:
        blocks = _build_sections_fallback_paragraph_headings(
            doc_df,
            title_fallback=title_fallback,
            header_footer_min_frac=header_footer_min_frac
        )

    if (
        not blocks
        or len(blocks) == 1
        or all(str(b.get("section_title", "")).strip() in {"", "PREAMBLE"} for b in blocks)
    ):
        return _full_document_block(doc_df, title_fallback="FULL_DOCUMENT")

    return blocks

def build_section_level_chunks(section_blocks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    chunks: List[Dict[str, Any]] = []

    for sec in section_blocks:
        chunk_text = sec["section_text"].strip()
        if not chunk_text:
            continue

        chunks.append({
            "doc_id": sec["doc_id"],
            "section_title": sec["section_title"],
            "page_start": sec["page_start"],
            "page_end": sec["page_end"],
            "chunk_page_start": sec["page_start"],
            "chunk_page_end": sec["page_end"],
            "chunk_index_in_section": 0,
            "n_words": len(chunk_text.split()),
            "chunk_text": chunk_text,
        })

    return chunks

def build_all_chunks(
    df_all_pages: pd.DataFrame,
    header_footer_min_frac: float = 0.4,
    min_keep_words: int = 20
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    all_sections: List[Dict[str, Any]] = []
    all_chunks: List[Dict[str, Any]] = []

    for doc_id, df_doc in df_all_pages.groupby("doc_id"):
        sec_blocks = build_sections_across_pages(
            df_doc,
            title_fallback="PREAMBLE",
            header_footer_min_frac=header_footer_min_frac
        )
        all_sections.extend(sec_blocks)

        doc_chunks = build_section_level_chunks(sec_blocks)

        for ch in doc_chunks:
            ch["chunk_id"] = (
                f"{doc_id}::"
                f"p{ch['chunk_page_start']:03d}-p{ch['chunk_page_end']:03d}::"
                f"s{ch['chunk_index_in_section']:04d}"
            )

        all_chunks.extend(doc_chunks)

    df_sections = pd.DataFrame([{
        "doc_id": s["doc_id"],
        "section_title": s["section_title"],
        "page_start": s["page_start"],
        "page_end": s["page_end"],
        "n_words": s["n_words"],
        "section_text": s["section_text"],
    } for s in all_sections])

    if not df_sections.empty:
        df_sections["section_num"] = df_sections["section_title"].str.extract(r"^\s*(\d+)\b", expand=False)

    df_chunks = pd.DataFrame(all_chunks)
    if not df_chunks.empty:
        df_chunks = df_chunks[df_chunks["n_words"] >= min_keep_words].reset_index(drop=True)

    return df_sections, df_chunks

In [ ]:
df_sections, df_chunks = build_all_chunks(df_pages)

print("Sections:", len(df_sections))
print("Chunks:", len(df_chunks))

Sections: 256
Chunks: 237


In [ ]:
if not df_chunks.empty:
    row = df_chunks.sample(1, random_state=42).iloc[0]
    print("DOC:", row["doc_id"])
    print("SECTION:", row["section_title"])
    print("PAGES:", row["page_start"], "-", row["page_end"])
    print("\nTEXT PREVIEW:\n", row["chunk_text"][:1200])
else:
    print("No chunks generated.")


DOC: Financial-Regulations
SECTION: 5. Status of the Financial Regulations
PAGES: 6 - 8

TEXT PREVIEW:
 5. Status of the Financial Regulations
5.1.
The Financial Regulations apply to all activities of QMUL; all staff (including
contractors and those employed by agencies), and all departments and
University subsidiaries.
5.2.
The Financial Regulations are subordinate to QMUL’s Charter and to any
restrictions imposed by the OfS Conditions of Registration.
5.3.
The purpose of these Financial Regulations is to provide control over the
totality of QMUL’s resources and to provide management with assurances that
the resources are being properly applied with the aim of achieving the goals of
QMUL’s Strategic Plan.
5.4.
Compliance with the Financial Regulations is compulsory for all staff
connected with QMUL and applies to all university income and business; all
departments and university subsidiary companies.
5.5.
Failure to comply with the Financial Regulations will be considered under
QMUL’s

In [ ]:
df_pages.to_csv(OUT_PATH, index=False)
df_sections.to_csv(SECTIONS_OUT, index=False)
df_chunks.to_csv(CHUNK_OUT, index=False)

print("Saved page-level data to:", OUT_PATH)
print("Saved sections to:", SECTIONS_OUT)
print("Saved chunks to:", CHUNK_OUT)


Saved page-level data to: /content/drive/MyDrive/IR_Qmary/ingested_pages.csv
Saved sections to: /content/drive/MyDrive/IR_Qmary/sections.csv
Saved chunks to: /content/drive/MyDrive/IR_Qmary/clean_chunks.csv


# 4. Indexing
## 4.1: Install Dependencies


In [ ]:
!pip install rank-bm25 scikit-learn ir_measures pandas numpy requests --break-system-packages -q

<frozen posixpath>:82: RuntimeWarning: coroutine 'ClientResponse.json' was never awaited


## 4.2: Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import string
import requests

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import ir_measures
from ir_measures import P, R, RR, nDCG, ScoredDoc, Qrel

print('All imports OK.')


All imports OK.


## 4.3: Load Chunks

Loads the single consolidated CSV (`clean_chunks.csv`) containing all 237 chunks across 20 documents.

**Duplicate chunk_id fix:** Multiple sections that fall on the same page range share the same
auto-generated chunk_id. We append `_s1`, `_s2`, etc. to guarantee uniqueness across the corpus.


In [ ]:
CSV_PATH = '/content/drive/MyDrive/IR_Qmary/clean_chunks.csv'

def load_chunks(csv_path):
    df = pd.read_csv(csv_path)
    df = df.rename(columns={
        'chunk_text':    'text',
        'section_title': 'source_title',})

    seen    = {}
    new_ids = []
    for cid in df['chunk_id']:
        if cid in seen:
            seen[cid] += 1
            new_ids.append(f'{cid}_s{seen[cid]}')
        else:
            seen[cid] = 0
            new_ids.append(cid)
    df['chunk_id'] = new_ids
    return df.to_dict(orient='records')

chunks = load_chunks(CSV_PATH)

n_docs = len(set(c['doc_id'] for c in chunks))
print(f'Loaded {len(chunks)} chunks from {n_docs} documents.\n')
print(f'{"DOCUMENT":<70} {"CHUNKS"}')
print('-' * 80)
for doc in sorted(set(c['doc_id'] for c in chunks)):
    count = sum(1 for c in chunks if c['doc_id'] == doc)
    print(f'{doc:<70} {count}')

Loaded 237 chunks from 20 documents.

DOCUMENT                                                               CHUNKS
--------------------------------------------------------------------------------
Anti-Bribery-and-Corruption-Policy-updated-June-2024                   11
Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2 24
Capital-Expenditure-Policy-2022-v1---approved                          20
Code-of-Practice-on-Free-Speech (1)                                    9
Conflict-of-Interest-Policy (2)                                        9
Criminal-Finances-Act-2017                                             1
Delegation-Framework (1)                                               8
Expenses-Policy                                                        14
Financial-Assistance-Policy                                            14
Financial-Regulations                                                  30
Fit-and-Proper-Person-Declaration (1)                           

## 4.4: Tokenisation, Stopwords & Query Expansion

**Tokenisation** (Manning et al., 2008): lowercase, remove punctuation, filter stopwords.

**Stopword removal** (Manning et al., 2008): 47 common words removed before indexing.

**Controlled synonym expansion** (Xu & Croft, 1996): domain-specific map bridges plain-English queries and formal policy vocabulary (e.g. *whistleblowing* → disclosure, reporting, concern).

**Query Expansion** (Xu & Croft, 1996; Rocchio, 1971): controlled synonym expansion using a
domain-specific map to reduce vocabulary mismatch between natural-language queries
and formal policy language.


In [ ]:
STOP_WORDS = {
    'a','an','the','is','it','in','on','at','to','for','of','and','or',
    'be','by','as','are','was','were','with','this','that','from','have',
    'has','had','not','but','they','their','them','its','if','any','all',
    'may','shall','will','can','do','does','such','each','which','who',
    'also','into','than','then','when','where','there','been','more','one',
    'under','about','would','should','must','upon','within','without',
    'other','these','those','between','through','during','before','after',
}

SYNONYM_MAP = {
    'whistleblowing':  ['disclosure', 'reporting', 'concern', 'malpractice'],
    'whistleblower':   ['reporter', 'discloser', 'complainant'],
    'concern':         ['complaint', 'allegation', 'issue', 'disclosure'],
    'fraud':           ['corruption', 'misconduct', 'deception', 'misappropriation'],
    'bribery':         ['corruption', 'inducement', 'kickback', 'improper'],
    'bribe':           ['inducement', 'payment', 'advantage', 'reward'],
    'malpractice':     ['wrongdoing', 'misconduct', 'irregularity', 'fraud'],
    'wrongdoing':      ['malpractice', 'misconduct', 'breach', 'fraud'],
    'laundering':      ['proceeds', 'criminal', 'finances', 'disclosure'],
    'money':           ['funds', 'proceeds', 'cash', 'financial'],
    'report':          ['raise', 'disclose', 'notify', 'submit'],
    'raise':           ['submit', 'report', 'disclose', 'notify'],
    'protection':      ['immunity', 'safeguard', 'detriment', 'reprisal'],
    'retaliation':     ['detriment', 'reprisal', 'victimisation'],
    'confidential':    ['anonymous', 'protected', 'private'],
    'anonymous':       ['confidential', 'unnamed', 'unidentified'],
    'conflict':        ['interest', 'bias', 'impartiality', 'loyalty'],
    'governance':      ['council', 'committee', 'oversight', 'accountability'],
    'delegation':      ['authority', 'responsibility', 'mandate', 'power'],
    'responsible':     ['accountable', 'designated', 'officer', 'role'],
    'approved':        ['authorised', 'sanctioned', 'permitted'],
    'expenditure':     ['spending', 'capital', 'costs', 'outlay'],
    'capital':         ['assets', 'investment', 'expenditure', 'buildings'],
    'investment':      ['assets', 'portfolio', 'funds', 'returns'],
    'treasury':        ['cash', 'liquidity', 'funds', 'banking'],
    'expenses':        ['claims', 'reimbursement', 'costs', 'allowances'],
    'financial':       ['monetary', 'fiscal', 'accounting', 'budgetary'],
    'assistance':      ['support', 'funding', 'bursary', 'award'],
    'refund':          ['compensation', 'reimbursement', 'repayment'],
    'risk':            ['threat', 'hazard', 'exposure', 'uncertainty'],
    'investigation':   ['inquiry', 'review', 'assessment', 'examination'],
    'legal':           ['legislative', 'statutory', 'act', 'law'],
    'policy':          ['procedure', 'guideline', 'regulation', 'rule'],
    'review':          ['appeal', 'reconsideration', 'reassessment'],
    'procedure':       ['process', 'steps', 'method', 'approach'],
    'apply':           ['scope', 'covers', 'applicable', 'includes'],
    'dress':           ['attire', 'uniform', 'gown', 'academic', 'robes'],
    'speech':          ['expression', 'debate', 'freedom', 'platform'],
    'gifts':           ['hospitality', 'entertainment', 'gratuity', 'benefit'],
    'employee':        ['worker', 'staff', 'personnel', 'contractor'],
    'student':         ['undergraduate', 'postgraduate', 'learner'],
}


def tokenise(text):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return [t for t in text.split() if t not in STOP_WORDS and len(t) > 1]

def expand_query(query):
    tokens   = tokenise(query)
    expanded = set(tokens)
    for t in tokens:
        if t in SYNONYM_MAP:
            for syn in SYNONYM_MAP[t]:
                expanded.update(tokenise(syn))
    return list(expanded)

print('Tokenisation & query expansion ready.')
test_q = 'What protection exists for whistleblowers?'
print(f'Example query : {test_q}')
print(f'Expanded      : {sorted(expand_query(test_q))}')

Tokenisation & query expansion ready.
Example query : What protection exists for whistleblowers?
Expanded      : ['detriment', 'exists', 'immunity', 'protection', 'reprisal', 'safeguard', 'what', 'whistleblowers']


# 5. Index and Models
## 5.1: Build BM25 and VSM Indexes

**BM25** (Robertson & Zaragoza, 2009):
$$\text{score}(q,d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{tf(t,d) \cdot (k_1+1)}{tf(t,d) + k_1\left(1 - b + b\cdot\frac{|d|}{\text{avgdl}}\right)}$$
Parameters tuned for short policy chunks: $k_1=1.2$ (lower TF saturation), $b=0.5$ (reduced length penalty).

**Section-title boosting** (Zhai & Lafferty, 2001): the section title is prepended twice to the
indexed text, acting as a soft field boost. This improves retrieval for sub-section queries
where query terms match the heading more than the body.

**VSM / TF-IDF** (Salton & Buckley, 1988):
$$\text{score}(q,d) = \cos\bigl(\text{tfidf}(q),\ \text{tfidf}(d)\bigr), \quad tf_{\text{scaled}} = \log(1 + tf)$$


In [ ]:
tokenised_corpus = [
    tokenise(c['source_title'] + ' ' + c['source_title'] + ' ' + c['text'])
    for c in chunks
]
bm25_index = BM25Okapi(tokenised_corpus, k1=1.2, b=0.5)

vsm_vectorizer = TfidfVectorizer(
    tokenizer=tokenise,
    token_pattern=None,
    sublinear_tf=True,
)
raw_corpus = [
    c['source_title'] + ' ' + c['source_title'] + ' ' + c['text']
    for c in chunks
]
vsm_matrix = vsm_vectorizer.fit_transform(raw_corpus)

print(f'BM25 index : {len(chunks)} chunks  |  vocab size : {len(bm25_index.idf)}')
print(f'VSM  index : {vsm_matrix.shape[0]} chunks  |  vocab size : {vsm_matrix.shape[1]}')

BM25 index : 237 chunks  |  vocab size : 5191
VSM  index : 237 chunks  |  vocab size : 5191


## 5.2: Retrieval Functions: BM25, VSM & RRF

**Reciprocal Rank Fusion** (Cormack et al., 2009):
$$\text{RRF}(d) = \sum_{r \in \text{ranklists}} \frac{1}{k + \text{rank}_r(d)}$$
where $k=60$ is a smoothing constant that prevents top-ranked documents from dominating the fused list.


In [ ]:
TOP_K           = 10
SCORE_THRESHOLD = 0.01


def retrieve_bm25(query, top_k=TOP_K):
    tokens = expand_query(query)
    scores = bm25_index.get_scores(tokens)
    ranked = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[i], 'score': float(scores[i]), 'rank': r+1}
            for r, i in enumerate(ranked) if scores[i] > 0]


def retrieve_vsm(query, top_k=TOP_K):
    expanded = ' '.join(expand_query(query))
    q_vec    = vsm_vectorizer.transform([expanded])
    sims     = cosine_similarity(q_vec, vsm_matrix).flatten()
    ranked   = np.argsort(sims)[::-1][:top_k]
    return [{**chunks[i], 'score': float(sims[i]), 'rank': r+1}
            for r, i in enumerate(ranked) if sims[i] > 0]


def reciprocal_rank_fusion(bm25_results, vsm_results, k=60, top_k=TOP_K):
    scores, chunk_store = {}, {}
    for i, hit in enumerate(bm25_results):
        cid = hit['chunk_id']
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + i + 1)
        chunk_store[cid] = hit
    for i, hit in enumerate(vsm_results):
        cid = hit['chunk_id']
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + i + 1)
        chunk_store.setdefault(cid, hit)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [{**chunk_store[cid], 'score': s, 'rank': r+1}
            for r, (cid, s) in enumerate(ranked)]


def retrieve(query, model='bm25', top_k=TOP_K):
    if   model == 'bm25': return retrieve_bm25(query, top_k)
    elif model == 'vsm':  return retrieve_vsm(query, top_k)
    elif model == 'rrf':
        return reciprocal_rank_fusion(
            retrieve_bm25(query, top_k),
            retrieve_vsm(query, top_k), top_k=top_k)
    else:
        raise ValueError(f"Unknown model '{model}'. Choose: 'bm25', 'vsm', 'rrf'.")


def retrieve_with_failsafe(query, model='bm25', threshold=SCORE_THRESHOLD):
    results  = retrieve(query, model=model)
    filtered = [r for r in results if r['score'] >= threshold]
    if not filtered:
        print(f'  [Failsafe] No relevant chunks for: "{query}"')
        return None
    return filtered


# Smoke test
test_query = 'What protection exists for whistleblowers?'
print(f'Smoke test: "{test_query}"\n')
for m in ['bm25', 'vsm', 'rrf']:
    res = retrieve(test_query, model=m)
    if res:
        print(f'  {m.upper():4s}  [{res[0]["source_title"]}]  score={res[0]["score"]:.4f}')
    else:
        print(f'  {m.upper():4s}  — no results')

Smoke test: "What protection exists for whistleblowers?"

  BM25  [5. Principles]  score=13.3614
  VSM   [2. Purpose]  score=0.1225
  RRF   [5. Principles]  score=0.0323


## 6. Queries and Relevance Judgements (QRELS)

**40 queries** — exactly 2 per document — one broad overview query and one specific
sub-section query per document.

Manual relevance grades follow the **Cranfield test collection paradigm** (Sanderson, 2013):
- `2` = highly relevant (directly and fully answers the query)
- `1` = relevant (partially addresses the query or provides useful context)
- `0` = not relevant

## 6.1: Queries

In [ ]:
QUERIES = {

    # Doc 01: Anti-Bribery-and-Corruption-Policy
    'q01': "What is bribery and what are QMUL's obligations under the Bribery Act 2010?",
    'q02': "What are QMUL's rules on accepting gifts, hospitality and facilitation payments?",

    # Doc 02: Anti-Money-Laundering-and-Criminal-Finances-Act-Policy
    'q03': "What is money laundering and what must QMUL employees do to prevent it?",
    'q04': "What is the procedure for reporting suspected money laundering and what does the MLRO do?",

    # Doc 03: Capital-Expenditure-Policy
    'q05': "What is the capital expenditure policy at QMUL and how are projects approved?",
    'q06': "How does QMUL distinguish between capital and revenue expenditure?",

    # Doc 04: Code-of-Practice-on-Free-Speech
    'q07': "What are QMUL's legal obligations and principles regarding free speech on campus?",
    'q08': "What does QMUL's free speech code of practice say about procedures for events and the use of external speakers?",

    # Doc 05: Conflict-of-Interest-Policy
    'q09': "How should a conflict of interest be identified and declared at QMUL?",
    'q10': "How does QMUL manage and resolve a declared conflict of interest?",

    # Doc 06: Criminal-Finances-Act-2017
    'q11': "What does the Criminal Finances Act 2017 require of QMUL and its staff?",
    'q12': "What counts as facilitating tax evasion under the Criminal Finances Act in a university?",

    # Doc 07: Delegation-Framework
    'q13': "What are the principles and structure of QMUL's delegation of authority framework?",
    'q14': "What finance-related responsibilities does QMUL Council hold under the delegation framework?",

    # Doc 08: Expenses-Policy
    'q15': "What expenses can QMUL staff claim and what is the process for doing so?",
    'q16': "What are the rules for claiming travel, mileage and accommodation expenses at QMUL?",

    # Doc 09: Financial-Assistance-Policy
    'q17': "What financial assistance is available to QMUL students and who is eligible?",
    'q18': "How are Financial Assistance Fund applications assessed for undergraduate home students?",

    # Doc 10: Financial-Regulations
    'q19': "What do QMUL's financial regulations cover and who is responsible for financial governance?",
    'q20': "What are QMUL staff responsibilities regarding expenditure and procurement?",

    # Doc 11: Fit-and-Proper-Person-Declaration
    'q21': "What is the fit and proper person declaration and who does it apply to at QMUL?",
    'q22': "What does the fit and proper person declaration form require and what is the sign-off procedure?",

    # Doc 12: Fraud-and-corruption-policy
    'q23': "What constitutes fraud at QMUL and what is the fraud response plan?",
    'q24': "What confidentiality protections exist for staff who report fraud at QMUL?",

    # Doc 13: Investment-Policy
    'q25': "What are QMUL's investment objectives and principles for managing its funds?",
    'q26': "What is QMUL's responsible investment policy and how are ESG criteria applied?",

    # Doc 14: Refund-and-Compensation-Policy
    'q27': "Under what circumstances can QMUL students request a refund or compensation?",
    'q28': "What does QMUL's refund and compensation policy say about its compensation plan and how payments are processed?",

    # Doc 15: Regulations-on-Official-Dress
    'q29': "What are QMUL's regulations on official dress and what does the policy state about academic ceremonial dress?",
    'q30': "What does QMUL's official dress policy say about the dress worn by senior officers including the Chair of Council?",

    # Doc 16: Risk-Management-Policy
    'q31': "What is QMUL's approach to identifying and managing institutional risks?",
    'q32': "What are the roles and responsibilities for risk management and how is the risk register maintained?",

    # Doc 17: Scheme-of-Delegation-of-Financial-Authority
    'q33': "What does the QMUL Scheme of Delegation of Financial Authority say about the framework for delegating financial authority?",
    'q34': "What are the specific financial spending limits delegated to heads of school and directors at QMUL?",

    # Doc 18: Standards-of-Business-Conduct
    'q35': "What standards of business conduct are QMUL staff expected to follow?",
    'q36': "What are the Seven Principles of public life and how should QMUL staff declare a conflict of interest?",

    # Doc 19: Treasury-Management-Policy
    'q37': "What are QMUL's treasury management practices and what principles guide the management of cash and investments?",
    'q38': "What counterparties is QMUL permitted to invest cash with and what are the monetary limits?",

    # Doc 20: WhistleblowingPolicy
    'q39': "How should a whistleblowing concern be raised and what protection does QMUL provide?",
    'q40': "What types of wrongdoing are covered by the QMUL whistleblowing policy and where can concerns be escalated externally?",}

QRELS = {
    'q01': {
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p001-p005::s0000': 2,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p005-p005::s0000': 1,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p005-p006::s0000': 1,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p008-p009::s0000': 1,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p009-p010::s0000': 1,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p006-p006::s0000': 0,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p010-p011::s0000': 0,
    },
    'q02': {
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p006-p007::s0000': 2,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p007-p007::s0000': 2,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p006-p006::s0000': 1,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p001-p005::s0000': 1,
        'Standards-of-Business-Conduct-updated-June-2024::p005-p006::s0000':      1,
        'Financial-Regulations::p014-p015::s0000': 1,
        'Anti-Bribery-and-Corruption-Policy-updated-June-2024::p005-p006::s0000': 0,
    },
    'q03': {
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p003-p004::s0000': 2,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p004-p004::s0000': 2,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p003-p003::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p005-p006::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p001-p002::s0000': 0,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p007-p008::s0000': 0,
    },
    'q04': {
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p005-p006::s0000': 2,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p006-p007::s0000': 2,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p005-p005::s0000': 2,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p013-p014::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p014-p014::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p007-p007::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p003-p004::s0000': 0,
    },
    'q05': {
        'Capital-Expenditure-Policy-2022-v1---approved::p003-p004::s0000': 2,
        'Capital-Expenditure-Policy-2022-v1---approved::p003-p003::s0000': 2,
        'Capital-Expenditure-Policy-2022-v1---approved::p001-p003::s0000': 1,
        'Capital-Expenditure-Policy-2022-v1---approved::p004-p004::s0000': 1,
        'Capital-Expenditure-Policy-2022-v1---approved::p008-p008::s0000': 1,
        'Capital-Expenditure-Policy-2022-v1---approved::p009-p010::s0000': 1,
        'Capital-Expenditure-Policy-2022-v1---approved::p004-p006::s0000': 0,
    },
    'q06': {
        'Capital-Expenditure-Policy-2022-v1---approved::p006-p008::s0000': 2,
        'Capital-Expenditure-Policy-2022-v1---approved::p004-p006::s0000': 2,
        'Capital-Expenditure-Policy-2022-v1---approved::p014-p015::s0000': 1,
        'Capital-Expenditure-Policy-2022-v1---approved::p011-p012::s0000': 1,
        'Capital-Expenditure-Policy-2022-v1---approved::p003-p003::s0000': 0,
        'Capital-Expenditure-Policy-2022-v1---approved::p001-p003::s0000': 0,
    },
    'q07': {
        'Code-of-Practice-on-Free-Speech (1)::p001-p002::s0000': 2,
        'Code-of-Practice-on-Free-Speech (1)::p001-p001::s0000': 2,
        'Code-of-Practice-on-Free-Speech (1)::p003-p003::s0000': 1,
        'Code-of-Practice-on-Free-Speech (1)::p003-p004::s0000': 1,
        'Code-of-Practice-on-Free-Speech (1)::p006-p006::s0000': 0,
    },
    'q08': {
        'Code-of-Practice-on-Free-Speech (1)::p004-p005::s0000': 2,
        'Code-of-Practice-on-Free-Speech (1)::p003-p004::s0000': 1,
        'Code-of-Practice-on-Free-Speech (1)::p003-p003::s0000': 1,
        'Code-of-Practice-on-Free-Speech (1)::p005-p005::s0000': 1,
        'Code-of-Practice-on-Free-Speech (1)::p001-p002::s0000': 0,
    },
    'q09': {
        'Conflict-of-Interest-Policy (2)::p002-p003::s0000': 2,
        'Conflict-of-Interest-Policy (2)::p003-p003::s0000': 2,
        'Conflict-of-Interest-Policy (2)::p006-p006::s0000': 1,
        'Conflict-of-Interest-Policy (2)::p007-p007::s0000': 1,
        'Conflict-of-Interest-Policy (2)::p001-p001::s0000': 1,
        'Conflict-of-Interest-Policy (2)::p005-p005::s0000': 0,
    },
    'q10': {
        'Conflict-of-Interest-Policy (2)::p003-p005::s0000': 2,
        'Conflict-of-Interest-Policy (2)::p007-p007::s0000': 1,
        'Conflict-of-Interest-Policy (2)::p003-p003::s0000': 1,
        'Conflict-of-Interest-Policy (2)::p005-p005::s0000': 1,
        'Conflict-of-Interest-Policy (2)::p002-p003::s0000': 0,
    },
    'q11': {
        'Criminal-Finances-Act-2017::p001-p001::s0000': 2,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p008-p008::s0000': 2,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p009-p010::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p010-p010::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p010-p011::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p001-p002::s0000': 0,
    },
    'q12': {
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p008-p009::s0000': 2,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p008-p008::s0000': 2,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p009-p010::s0000': 1,
        'Criminal-Finances-Act-2017::p001-p001::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p010-p010::s0000': 1,
        'Anti-Money-Laundering-and-Criminal-Finances-Act-Policy-updated-21.2.18-v2::p003-p004::s0000': 0,
    },
    'q13': {
        'Delegation-Framework (1)::p001-p003::s0000': 2,
        'Delegation-Framework (1)::p001-p001::s0000': 2,
        'Delegation-Framework (1)::p003-p004::s0000': 1,
        'Delegation-Framework (1)::p004-p005::s0000': 1,
        'Delegation-Framework (1)::p004-p004::s0000': 1,
        'Delegation-Framework (1)::p006-p021::s0000': 0,
    },
    'q14': {
        'Delegation-Framework (1)::p003-p004::s0000': 2,
        'Delegation-Framework (1)::p005-p006::s0000': 2,
        'Delegation-Framework (1)::p003-p003::s0000': 1,
        'Delegation-Framework (1)::p006-p021::s0000': 1,
        'Financial-Regulations::p008-p009::s0000': 1,
        'Delegation-Framework (1)::p001-p001::s0000': 0,
    },
    'q15': {
        'Expenses-Policy::p006-p007::s0000': 2,
        'Expenses-Policy::p003-p004::s0000': 1,
        'Expenses-Policy::p004-p005::s0000': 1,
        'Expenses-Policy::p001-p002::s0000': 1,
        'Expenses-Policy::p003-p003::s0000': 0,
    },
    'q16': {
        'Expenses-Policy::p007-p010::s0000': 2,
        'Expenses-Policy::p010-p012::s0000': 2,
        'Expenses-Policy::p012-p013::s0000': 2,
        'Expenses-Policy::p006-p007::s0000': 1,
        'Expenses-Policy::p003-p004::s0000': 0,
    },
    'q17': {
        'Financial-Assistance-Policy::p002-p003::s0000': 2,
        'Financial-Assistance-Policy::p002-p002::s0000': 2,
        'Financial-Assistance-Policy::p001-p001::s0000': 1,
        'Financial-Assistance-Policy::p007-p007::s0000': 1,
        'Financial-Assistance-Policy::p010-p010::s0000': 1,
        'Financial-Assistance-Policy::p003-p003::s0000': 0,
    },
    'q18': {
        'Financial-Assistance-Policy::p003-p006::s0000': 2,
        'Financial-Assistance-Policy::p002-p003::s0000': 1,
        'Financial-Assistance-Policy::p011-p012::s0000': 1,
        'Financial-Assistance-Policy::p003-p003::s0000': 1,
        'Financial-Assistance-Policy::p010-p010::s0000': 1,
        'Financial-Assistance-Policy::p001-p001::s0000': 0,
    },
    'q19': {
        'Financial-Regulations::p004-p004::s0000': 2,
        'Financial-Regulations::p006-p008::s0000': 2,
        'Financial-Regulations::p008-p009::s0000': 1,
        'Financial-Regulations::p009-p013::s0000': 1,
        'Financial-Regulations::p004-p005::s0000': 1,
        'Financial-Regulations::p001-p003::s0000': 0,
    },
    'q20': {
        'Financial-Regulations::p029-p032::s0000': 2,
        'Financial-Regulations::p009-p013::s0000': 1,
        'Financial-Regulations::p015-p016::s0000': 1,
        'Financial-Regulations::p018-p019::s0000': 1,
        'Financial-Regulations::p016-p018::s0000': 1,
        'Financial-Regulations::p001-p003::s0000': 0,
    },
    'q21': {
        'Fit-and-Proper-Person-Declaration (1)::p001-p001::s0000': 2,
        'Fit-and-Proper-Person-Declaration (1)::p001-p001::s0000_s1': 1,
        'Fit-and-Proper-Person-Declaration (1)::p001-p002::s0000': 1,
        'Fit-and-Proper-Person-Declaration (1)::p003-p003::s0000': 1,
        'Financial-Regulations::p009-p013::s0000': 0,
    },
    'q22': {
        'Fit-and-Proper-Person-Declaration (1)::p003-p003::s0000': 2,
        'Fit-and-Proper-Person-Declaration (1)::p001-p001::s0000_s1': 2,
        'Fit-and-Proper-Person-Declaration (1)::p001-p002::s0000': 1,
        'Fit-and-Proper-Person-Declaration (1)::p001-p001::s0000': 0,
    },
    'q23': {
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p004-p004::s0000': 2,
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p006-p011::s0000': 2,
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p003-p004::s0000': 1,
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p005-p006::s0000': 1,
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p004-p005::s0000': 1,
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p001-p002::s0000': 0,
    },
    'q24': {
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p005-p005::s0000': 2,
        'WhistleblowingPolicy (1)::p002-p003::s0000': 1,
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p005-p006::s0000': 1,
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p006-p011::s0000': 1,
        'Fraud-and-corruption-policy-updated-June-2024 (2)::p003-p004::s0000': 0,
    },
    'q25': {
        'Investment-Policy::p001-p001::s0000': 2,
        'Investment-Policy::p002-p002::s0000': 2,
        'Investment-Policy::p002-p003::s0000': 1,
        'Investment-Policy::p001-p002::s0000': 1,
        'Investment-Policy::p006-p006::s0000': 1,
        'Investment-Policy::p006-p009::s0000': 0,
    },
    'q26': {
        'Investment-Policy::p003-p005::s0000': 2,
        'Investment-Policy::p005-p006::s0000': 1,
        'Investment-Policy::p001-p001::s0000': 1,
        'Investment-Policy::p006-p009::s0000': 1,
        'Investment-Policy::p002-p002::s0000': 0,
    },
    'q27': {
        'Refund-and-Compensation-Policy::p003-p003::s0000': 2,
        'Refund-and-Compensation-Policy::p001-p001::s0000': 2,
        'Refund-and-Compensation-Policy::p003-p004::s0000': 1,
        'Refund-and-Compensation-Policy::p002-p002::s0000': 1,
        'Refund-and-Compensation-Policy::p005-p006::s0000': 0,
    },
    'q28': {
        'Refund-and-Compensation-Policy::p004-p005::s0000': 2,
        'Refund-and-Compensation-Policy::p005-p005::s0000': 2,
        'Refund-and-Compensation-Policy::p003-p004::s0000': 1,
        'Refund-and-Compensation-Policy::p003-p003::s0000': 1,
        'Refund-and-Compensation-Policy::p001-p001::s0000': 0,
    },
    'q29': {
        'Regulations-on-Official-Dress (2)::p001-p001::s0000': 2,
        'Regulations-on-Official-Dress (2)::p001-p001::s0000_s1': 1,
        'Regulations-on-Official-Dress (2)::p001-p001::s0000_s2': 1,
        'Regulations-on-Official-Dress (2)::p001-p001::s0000_s3': 1,
        'Regulations-on-Official-Dress (2)::p001-p001::s0000_s4': 1,
        'Regulations-on-Official-Dress (2)::p001-p001::s0000_s6': 0,
    },
    'q30': {
        'Regulations-on-Official-Dress (2)::p001-p001::s0000_s1': 2,
        'Regulations-on-Official-Dress (2)::p001-p001::s0000_s3': 2,
        'Regulations-on-Official-Dress (2)::p001-p001::s0000': 1,
        'Regulations-on-Official-Dress (2)::p001-p001::s0000_s2': 1,
        'Regulations-on-Official-Dress (2)::p001-p001::s0000_s4': 0,
    },
    'q31': {
        'Risk-Management-Policy---September-2024 (2)::p002-p002::s0000': 2,
        'Risk-Management-Policy---September-2024 (2)::p001-p001::s0000': 2,
        'Risk-Management-Policy---September-2024 (2)::p002-p002::s0000_s1': 2,
        'Risk-Management-Policy---September-2024 (2)::p002-p004::s0000': 1,
        'Risk-Management-Policy---September-2024 (2)::p001-p001::s0000_s1': 1,
        'Risk-Management-Policy---September-2024 (2)::p005-p005::s0000': 1,
        'Financial-Regulations::p015-p016::s0000': 1,
        'Risk-Management-Policy---September-2024 (2)::p006-p006::s0000': 0,
    },
    'q32': {
        'Risk-Management-Policy---September-2024 (2)::p002-p004::s0000': 2,
        'Risk-Management-Policy---September-2024 (2)::p005-p005::s0000': 2,
        'Risk-Management-Policy---September-2024 (2)::p006-p006::s0000': 2,
        'Risk-Management-Policy---September-2024 (2)::p002-p002::s0000': 1,
        'Risk-Management-Policy---September-2024 (2)::p001-p001::s0000': 0,
    },
    'q33': {
        'Scheme-of-Delegation-of-Financial-Authority::p001-p009::s0000': 2,
        'Delegation-Framework (1)::p005-p006::s0000': 1,
        'Delegation-Framework (1)::p006-p021::s0000': 1,
        'Scheme-of-Delegation-of-Financial-Authority::p010-p010::s0000': 1,
        'Financial-Regulations::p024-p025::s0000': 1,
        'Delegation-Framework (1)::p001-p003::s0000': 0,
    },
    'q34': {
        'Scheme-of-Delegation-of-Financial-Authority::p001-p009::s0000': 2,
        'Scheme-of-Delegation-of-Financial-Authority::p010-p010::s0000': 2,
        'Delegation-Framework (1)::p005-p006::s0000': 1,
        'Delegation-Framework (1)::p006-p021::s0000': 1,
        'Financial-Regulations::p024-p025::s0000': 0,
    },
    'q35': {
        'Standards-of-Business-Conduct-updated-June-2024::p003-p004::s0000': 2,
        'Standards-of-Business-Conduct-updated-June-2024::p003-p003::s0000': 2,
        'Standards-of-Business-Conduct-updated-June-2024::p004-p005::s0000': 1,
        'Standards-of-Business-Conduct-updated-June-2024::p004-p004::s0000': 1,
        'Standards-of-Business-Conduct-updated-June-2024::p001-p002::s0000': 0,
    },
    'q36': {
        'Standards-of-Business-Conduct-updated-June-2024::p006-p020::s0000': 2,
        'Financial-Regulations::p041-p041::s0000': 2,
        'Standards-of-Business-Conduct-updated-June-2024::p004-p004::s0000': 1,
        'Standards-of-Business-Conduct-updated-June-2024::p003-p004::s0000': 1,
        'Standards-of-Business-Conduct-updated-June-2024::p002-p002::s0000': 0,
        'Standards-of-Business-Conduct-updated-June-2024::p001-p002::s0000': 0,
    },
    'q37': {
        'Treasury-Management-Policy::p005-p005::s0000': 2,
        'Treasury-Management-Policy::p005-p005::s0000_s1': 2,
        'Treasury-Management-Policy::p003-p003::s0000_s3': 2,
        'Treasury-Management-Policy::p005-p016::s0000': 1,
        'Treasury-Management-Policy::p003-p005::s0000': 1,
        'Treasury-Management-Policy::p003-p003::s0000': 1,
        'Treasury-Management-Policy::p001-p002::s0000': 1,
        'Treasury-Management-Policy::p019-p021::s0000': 0,
    },
    'q38': {
        'Treasury-Management-Policy::p018-p018::s0000': 2,
        'Treasury-Management-Policy::p017-p017::s0000': 1,
        'Treasury-Management-Policy::p005-p016::s0000': 1,
        'Treasury-Management-Policy::p005-p005::s0000': 1,
        'Treasury-Management-Policy::p003-p003::s0000': 0,
    },
    'q39': {
        'WhistleblowingPolicy (1)::p004-p006::s0000': 2,
        'WhistleblowingPolicy (1)::p002-p003::s0000': 2,
        'WhistleblowingPolicy (1)::p003-p004::s0000': 1,
        'WhistleblowingPolicy (1)::p001-p002::s0000': 1,
        'WhistleblowingPolicy (1)::p001-p001::s0000': 1,
        'WhistleblowingPolicy (1)::p008-p009::s0000': 0,
    },
    'q40': {
        'WhistleblowingPolicy (1)::p001-p002::s0000': 2,
        'WhistleblowingPolicy (1)::p007-p008::s0000': 2,
        'WhistleblowingPolicy (1)::p004-p006::s0000': 1,
        'WhistleblowingPolicy (1)::p002-p003::s0000': 1,
        'WhistleblowingPolicy (1)::p001-p001::s0000': 0,
    },
}

with open('queries.json', 'w') as f:
    json.dump(QUERIES, f, indent=2)
with open('qrels.json', 'w') as f:
    json.dump(QRELS, f, indent=2)

n_judgements = sum(len(v) for v in QRELS.values())
print(f'Defined {len(QUERIES)} queries (2 per document, 20 documents).')
print(f'Defined {n_judgements} total relevance judgements across {len(QRELS)} queries.')
print(f'Average judgements per query: {n_judgements / len(QRELS):.1f}')
print('Saved queries.json and qrels.json.')

Defined 40 queries (2 per document, 20 documents).
Defined 230 total relevance judgements across 40 queries.
Average judgements per query: 5.8
Saved queries.json and qrels.json.


# 7. Evaluation Stage-1: Classical IR Evaluation

Evaluates BM25, VSM, and RRF using the **Cranfield paradigm** (Sanderson, 2013).
Reads from 'qrels.json' which should be in the same folder as this notebook.
**Metrics** (all at cutoff 10):
- **P@10** — Precision: $P@k = \frac{|\{\text{relevant docs}\} \cap \{\text{retrieved docs}\}|}{k}$
- **R@10** — Recall: $R@k = \frac{|\{\text{relevant docs}\} \cap \{\text{retrieved docs}\}|}{|\{\text{relevant docs}\}|}$
- **MRR@10** — Mean Reciprocal Rank: $\text{MRR} = \frac{1}{|Q|}\sum_{q}\frac{1}{\text{rank}_q}$
- **nDCG@10** — Normalised DCG: $\text{nDCG} = \frac{\text{DCG}}{\text{IDCG}}$, $\text{DCG} = \sum_{i=1}^{k}\frac{\text{rel}_i}{\log_2(i+1)}$


In [ ]:
def load_qrels(path='qrels.json'):
    with open(path) as f:
        raw = json.load(f)
    return [
        Qrel(query_id=qid, doc_id=did, relevance=grade)
        for qid, docs in raw.items()
        for did, grade in docs.items()
    ]


def build_run(queries, model):
    run = []
    for qid, qtext in queries.items():
        for hit in retrieve(qtext, model=model):
            run.append(ScoredDoc(
                query_id=qid,
                doc_id=hit['chunk_id'],
                score=float(hit['score']),
            ))
    return run


def evaluate_all_models():
    qrels   = load_qrels()
    metrics = [P@10, R@10, RR@10, nDCG@10]
    rows    = []

    for model in ['bm25', 'vsm', 'rrf']:
        run        = build_run(QUERIES, model)
        aggregated = ir_measures.calc_aggregate(metrics, qrels, run)
        row        = {str(m): round(float(v), 4) for m, v in aggregated.items()}
        row['Model'] = model.upper()
        rows.append(row)
        print(f'  {model.upper()} evaluated.')

    df = pd.DataFrame(rows).set_index('Model')

    rename = {}
    for col in df.columns:
        if   'RR@10'   in col: rename[col] = 'MRR@10'
        elif 'R@10'    in col: rename[col] = 'R@10'
        elif 'P@10'    in col: rename[col] = 'P@10'
        elif 'nDCG@10' in col: rename[col] = 'nDCG@10'
    df = df.rename(columns=rename)

    print('\n=== Stage 1: Classical IR Evaluation ===')
    print(df.to_string())
    df.to_csv('results_ir_evaluation.csv')
    print('\nSaved → results_ir_evaluation.csv')
    return df


print('Running Stage 1 IR Evaluation...')
ir_results = evaluate_all_models()


Running Stage 1 IR Evaluation...
  BM25 evaluated.
  VSM evaluated.
  RRF evaluated.

=== Stage 1: Classical IR Evaluation ===
        P@10  MRR@10    R@10  nDCG@10
Model                                
BM25   0.280  0.7858  0.6086   0.5886
VSM    0.295  0.6725  0.6320   0.5460
RRF    0.295  0.7609  0.6390   0.5884

Saved → results_ir_evaluation.csv


## 8. RAG Generation via OpenAI API

Uses **GPT-4o-mini** via the official OpenAI Python SDK.  
Reliable, no monthly credit exhaustion, OpenAI-compatible interface.  
Temperature=0.1, max_tokens=500 for complete policy answers.

**Setup:** paste your OpenAI API key below. Get one at https://platform.openai.com/api-keys

In [ ]:
!pip install openai

In [ ]:
import os
from openai import OpenAI

In [ ]:
OPENAI_API_KEY  = 'your_openai_api_key'   # <- update this
OPENAI_MODEL    = 'gpt-4o-mini'
_openai_client  = OpenAI(api_key=OPENAI_API_KEY)

MAX_NEW_TOKENS     = 500
MAX_CONTEXT_CHUNKS = 5
MAX_CHUNK_CHARS    = 1200


def build_prompt(query, selected_chunks):
    context_parts = []
    for i, chunk in enumerate(selected_chunks, start=1):
        text = chunk['text'][:MAX_CHUNK_CHARS]
        context_parts.append(
            f"[{i}] {chunk['source_title']} "
            f"(pages {chunk['page_start']}-{chunk['page_end']})\n{text}\n"
        )
    return (
        'You are an expert assistant for QMUL policy documents. '
        'Answer using ONLY the context below. '
        'Cite sources with [1], [2], etc. '
        'If the answer is not in the context, say: '
        '"I cannot find this information in the provided policy documents."\n\n'
        f'Context:\n{chr(10).join(context_parts)}\n'
        f'Question: {query}\nAnswer:'
    )


def call_llm(prompt):
    try:
        response = _openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=MAX_NEW_TOKENS,
            temperature=0.1,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f'Error: {e}'


def generate_answer(query, model='bm25'):
    """Full RAG pipeline: retrieve -> prompt -> LLM -> return answer + citations."""
    results = retrieve_with_failsafe(query, model=model)
    if results is None:
        return {
            'query':           query,
            'answer':          'No information found.',
            'citations':       [],
            'chunks_used':     [],
            'retrieval_model': model,
        }
    selected  = results[:MAX_CONTEXT_CHUNKS]
    prompt    = build_prompt(query, selected)
    answer    = call_llm(prompt)
    citations = [
        {
            'number':       i + 1,
            'chunk_id':     c['chunk_id'],
            'source_title': c['source_title'],
            'pages':        f"{c['page_start']}-{c['page_end']}",
            'document':       c['doc_id'],       # e.g. "WhistleblowingPolicy (1)"
        }
        for i, c in enumerate(selected)
    ]
    return {
        'query':           query,
        'answer':          answer,
        'citations':       citations,
        'chunks_used':     selected,
        'retrieval_model': model,
    }


# Quick test
print('Testing RAG pipeline (BM25 + GPT-4o-mini)...')
test = generate_answer('What protection exists for whistleblowers at QMUL?', model='bm25')
print(f'Q: {test["query"]}')
print(f'A: {test["answer"][:400]}...')
print('\nCitations:')
for c in test['citations']:
    print(f"  [{c['number']}] {c['source_title']} (pp. {c['pages']}) - {c['document']}")

Testing RAG pipeline (BM25 + GPT-4o-mini)...
Q: What protection exists for whistleblowers at QMUL?
A: Whistleblowers at QMUL are protected under the Public Interest Disclosure Act 1998, which ensures they do not suffer any detrimental treatment, including dismissal, disciplinary action, threats, or other unfavourable treatment as a result of raising concerns in the correct way [2]. The QMUL policy also states that whistleblowers must not experience any detrimental treatment for raising concerns in...

Citations:
  [1] 5. Principles (pp. 2-3) - WhistleblowingPolicy (1)
  [2] 3. Legislative context (pp. 1-1) - WhistleblowingPolicy (1)
  [3] 2. Purpose and Legislative Context (pp. 3-3) - Expenses-Policy
  [4] 10. Bribery concerns (pp. 8-9) - Anti-Bribery-and-Corruption-Policy-updated-June-2024
  [5] 9. Related policies and sources of advice (pp. 7-8) - WhistleblowingPolicy (1)


## 9. Answer Cache

Saves all LLM answers to `answer_cache.json` on disk.  
If the evaluation is interrupted, it resumes without re-calling the API.  
Call `clear_cache()` to force a full fresh run.

In [ ]:
CACHE_FILE = 'answer_cache.json'

def _load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE) as f:
            return json.load(f)
    return {}

def _save_cache(cache):
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

_answer_cache = _load_cache()
print(f'Answer cache loaded: {len(_answer_cache)} entries')


def generate_answer_cached(query, model='bm25'):
    cache_key = f'{model}::{query}'
    if cache_key in _answer_cache:
        return _answer_cache[cache_key]
    result = generate_answer(query, model=model)
    answer = result.get('answer', '')
    if not answer.startswith('Error:') and answer != 'No information found.':
        _answer_cache[cache_key] = result
        _save_cache(_answer_cache)
    return result


def clear_cache():
    global _answer_cache
    _answer_cache = {}
    if os.path.exists(CACHE_FILE):
        os.remove(CACHE_FILE)
    print('Cache cleared.')


print('Cache ready. Use generate_answer_cached() in the evaluation loop.')
print('Call clear_cache() to force a full re-run from scratch.')

Answer cache loaded: 40 entries
Cache ready. Use generate_answer_cached() in the evaluation loop.
Call clear_cache() to force a full re-run from scratch.


## 10. Evaluation Stage 2: RAGAS Setup (Gemini 2.5 Flash as Judge)

**Gemini 2.5 Flash** is wired as the RAGAS judge LLM and embedding model.  
Using a different model as judge than generator (GPT-4o-mini) ensures independent evaluation.

**Metrics** (Es et al., 2024) — no `ground_truth` required:
- **Faithfulness** — LLM decomposes answer into atomic claims, judges each against the context
- **ResponseRelevancy** — LLM generates reverse questions from the answer, checks embedding similarity
- **ContextUtilization** — LLM judges how well the retrieved context was actually used



In [ ]:
!pip  install ragas

In [ ]:
!pip install langchain_google_genai
!pip install langchain_openai

In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, ResponseRelevancy, ContextUtilization
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import OpenAIEmbeddings

GEMINI_API_KEY = 'your_gemini_api_key'   # <- update this
os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY

ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model='text-embedding-3-small', api_key=OPENAI_API_KEY)
)

RAGAS_METRICS = [
    Faithfulness(),
    ResponseRelevancy(),
    ContextUtilization(),
]

print('RAGAS configured successfully.')
print('  Generation LLM : GPT-4o-mini (OpenAI)')
print('  Judge LLM      : gemini-2.5-flash (Google)')
print('  Embeddings     : text-embedding-3-small (OpenAI)')
print('  Metrics        : Faithfulness, ResponseRelevancy, ContextUtilization ')

/tmp/ipykernel_1690/43467453.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy, ContextUtilization
/tmp/ipykernel_1690/43467453.py:2: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy, ContextUtilization
/tmp/ipykernel_1690/43467453.py:2: DeprecationWarning: Importing ContextUtilization from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextUtilization
  from ragas.metrics import Faithfulness, ResponseRelevancy, Context

RAGAS configured successfully.
  Generation LLM : GPT-4o-mini (OpenAI)
  Judge LLM      : gemini-2.5-flash (Google)
  Embeddings     : text-embedding-3-small (OpenAI)
  Metrics        : Faithfulness, ResponseRelevancy, ContextUtilization 


/tmp/ipykernel_1690/43467453.py:11: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
/tmp/ipykernel_1690/43467453.py:15: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


## Single query RAGAS Smoke Test
Single query RAGAS test   
Run this to verify RAGAS is working before the full 40-query evaluation.  
Uses q01 with BM25. Should complete in ~30 seconds.


In [ ]:
test_qid   = 'q01'
test_query = QUERIES[test_qid]
test_model = 'bm25'

print(f'Testing RAGAS on one query: {test_qid}')
print(f'Query: {test_query}\n')

result   = generate_answer_cached(test_query, model=test_model)
answer   = result['answer']
selected = result['chunks_used']

print(f'Answer: {answer[:300]}...\n')

test_sample = [{
    'user_input':         test_query,
    'response':           answer,
    'retrieved_contexts': [c['text'] for c in selected],
}]

test_dataset = EvaluationDataset.from_list(test_sample)
test_scores  = evaluate(
    dataset=test_dataset,
    metrics=RAGAS_METRICS,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
).to_pandas()

print('RAGAS scores for this query:')
for col in test_scores.columns:
    if col not in ['user_input', 'response', 'retrieved_contexts']:
        val = test_scores[col].iloc[0]
        try:
            print(f'  {col:<28}: {float(val):.4f}')
        except (TypeError, ValueError):
            print(f'  {col:<28}: {val}')

print('\nColumn names produced by RAGAS:', list(test_scores.columns))
print('\nIf scores look correct, run next cell for the full evaluation.')

Testing RAGAS on one query: q01
Query: What is bribery and what are QMUL's obligations under the Bribery Act 2010?

Answer: Bribery is defined as the dishonest offering, giving, soliciting, or acceptance of an inducement or reward that may influence the action of any person. It involves two or more people engaging in this dishonest conduct [2].

Under the Bribery Act 2010, it is an offence for an employee of QMUL to acce...



Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/lib/python3.12/pathlib.py:407: RuntimeWarning: coroutine 'ClientResponse.json' was never awaited
  def _load_parts(self):


RAGAS scores for this query:
  faithfulness                : 0.7143
  answer_relevancy            : 0.9348
  context_utilization         : 0.7500

Column names produced by RAGAS: ['user_input', 'retrieved_contexts', 'response', 'faithfulness', 'answer_relevancy', 'context_utilization']

If scores look correct, run next cell for the full evaluation.


## 11. Run RAGAS Evaluation

Runs the full pipeline for all **40 queries × 3 retrieval models = 120 LLM calls**.  
Answers are generated by GPT-4o-mini, then scored by Gemini 2.5 Flash via RAGAS.  
Estimated cost: ~\$0.05 generation + ~\$0.03 judging. Estimated time: ~10 minutes.

In [ ]:
def run_ragas_evaluation():
    all_rows = []

    for model in ['bm25', 'vsm', 'rrf']:
        print(f'\nCollecting answers for {model.upper()}...')
        samples = []
        skipped = []
        valid   = []

        for qid, qtext in QUERIES.items():
            print(f'  {qid}...', end=' ', flush=True)

            result   = generate_answer_cached(qtext, model=model)
            answer   = result['answer']
            selected = result['chunks_used']

            refusal_phrase = 'i cannot find this information in the provided policy documents'
            is_refusal     = answer.lower().strip().startswith(refusal_phrase[:40])

            if (answer.startswith('Error:')
                    or answer == 'No information found.'
                    or is_refusal
                    or not selected):
                print(f'skipped ({answer[:50]})')
                skipped.append(qid)
                continue

            samples.append({
                'user_input':         qtext,
                'response':           answer,
                'retrieved_contexts': [c['text'] for c in selected],
            })
            valid.append(qid)
            print('OK')

        if not samples:
            print(f'  No valid answers for {model.upper()}, skipping.')
            continue

        print(f'\n  Running RAGAS on {len(samples)} answers ({len(skipped)} skipped)...')

        eval_dataset = EvaluationDataset.from_list(samples)
        result = evaluate(
            dataset=eval_dataset,
            metrics=RAGAS_METRICS,
            llm=ragas_llm,
            embeddings=ragas_embeddings,
        )
        scores = result.to_pandas()
        scores['model']    = model.upper()
        scores['query_id'] = valid
        scores.to_csv(f'results_ragas_{model}.csv', index=False)

        metric_cols = [c for c in scores.columns
                       if c in ['faithfulness', 'answer_relevancy', 'context_utilization']]
        agg = scores[metric_cols].mean()

        print(f'\n  {model.upper()} RAGAS averages ({len(samples)}/40 answered):')
        for col in metric_cols:
            print(f'    {col:<28}: {agg[col]:.4f}')

        row = {'Model': model.upper()}
        row.update({col: round(float(agg[col]), 4) for col in metric_cols})
        all_rows.append(row)

    summary = pd.DataFrame(all_rows).set_index('Model')
    summary.to_csv('results_ragas_comparison.csv')
    print('\n=== Stage 2: Real RAGAS Evaluation (All Models) ===')
    print(summary.to_string())
    print('\nSaved -> results_ragas_comparison.csv')
    return summary


print('Running Stage 2 RAGAS Evaluation...')
rag_results = run_ragas_evaluation()

Running Stage 2 RAGAS Evaluation...

  q01... OK
  q02... OK
  q03... OK
  q04... OK
  q05... OK
  q06... OK
  q07... OK
  q08... OK
  q09... OK
  q10... OK
  q11... OK
  q12... OK
  q13... OK
  q14... OK
  q15... OK
  q16... OK
  q17... OK
  q18... OK
  q19... OK
  q20... OK
  q21... OK
  q22... OK
  q23... OK
  q24... OK
  q25... OK
  q26... OK
  q27... OK
  q28... OK
  q29... OK
  q30... skipped (I cannot find this information in the provided pol)
  q31... OK
  q32... OK
  q33... OK
  q34... skipped (I cannot find this information in the provided pol)
  q35... OK
  q36... OK
  q37... OK
  q38... OK
  q39... OK
  q40... OK

  Running RAGAS on 38 answers (2 skipped)...


Evaluating:   0%|          | 0/114 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[6]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[12]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[15]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[18]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[21]: TimeoutError()
/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py:214: RuntimeWarning: coroutine 'ClientResponse.json' was never awaited
  def send_multipart(self, *args, **kwargs):
ERROR:ragas.executor:Exception raised in Job[24]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[30]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[42]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[59]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[63]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[72]: TimeoutError()
/usr/lib/python3.12/asyncio/futures.py:367: 


  BM25 RAGAS averages (38/40 answered):
    faithfulness                : 0.9652
    answer_relevancy            : 0.9084
    context_utilization         : 0.6650

  q01... OK
  q02... OK
  q03... OK
  q04... OK
  q05... OK
  q06... OK
  q07... OK
  q08... skipped (I cannot find this information in the provided pol)
  q09... OK
  q10... OK
  q11... OK
  q12... OK
  q13... OK
  q14... OK
  q15... OK
  q16... OK
  q17... OK
  q18... OK
  q19... OK
  q20... 

/usr/local/lib/python3.12/dist-packages/h11/_state.py:319: RuntimeWarning: coroutine 'ClientResponse.json' was never awaited
  start_states = dict(self.states)


...

ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[18]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[42]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[45]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[60]: TimeoutError()



  VSM RAGAS averages (38/40 answered):
    faithfulness                : 0.9431
    answer_relevancy            : 0.9120
    context_utilization         : 0.5910

  q01... OK
  q02... OK
  q03... OK
  q04... OK
  q05... OK
  q06... OK
  q07... OK
  q08... OK
  q09... OK
  q10... OK
  q11... OK
  q12... OK
  q13... OK
  q14... OK
  q15... OK
  q16... OK
  q17... OK
  q18... OK
  q19... OK
  q20... OK
  q21... OK
  q22... OK
  q23... OK
  q24... OK
  q25... OK
  q26... OK
  q27... OK
  q28... OK
  q29... OK
  q30... OK
  q31... OK
  q32... OK
  q33... OK
  q34... skipped (I cannot find this information in the provided pol)
  q35... OK
  q36... OK
  q37... OK
  q38... OK
  q39... OK
  q40... OK

  Running RAGAS on 39 answers (1 skipped)...


Evaluating:   0%|          | 0/117 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: RuntimeWarning: coroutine 'ClientResponse.json' was never awaited
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)



  RRF RAGAS averages (39/40 answered):
    faithfulness                : 0.9735
    answer_relevancy            : 0.9261
    context_utilization         : 0.6293

=== Stage 2: Real RAGAS Evaluation (All Models) ===
       faithfulness  answer_relevancy  context_utilization
Model                                                     
BM25         0.9652            0.9084               0.6650
VSM          0.9431            0.9120               0.5910
RRF          0.9735            0.9261               0.6293

Saved -> results_ragas_comparison.csv


## 12. Final overall summary

In [ ]:
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)

print('\n--- Stage 1: Classical IR Evaluation ---')
print('Metrics: P@10, R@10, MRR@10, nDCG@10  (Sanderson, 2013)')
print(ir_results.to_string())

print('\n--- Stage 2: Real RAGAS Evaluation ---')
print('Generation LLM : GPT-4o-mini (OpenAI)')
print('Judge LLM      : Gemini 2.5 Flash  (Es et al., 2024)')
print('Metrics        : Faithfulness, ResponseRelevancy, ContextUtilization')
print(rag_results.to_string())

print('\nOutput files saved:')
for f in [
    'results_ir_evaluation.csv',
    'results_ragas_bm25.csv',
    'results_ragas_vsm.csv',
    'results_ragas_rrf.csv',
    'results_ragas_comparison.csv',
]:
    print(f'  {f}')

FINAL RESULTS SUMMARY

--- Stage 1: Classical IR Evaluation ---
Metrics: P@10, R@10, MRR@10, nDCG@10  (Sanderson, 2013)
        P@10  MRR@10    R@10  nDCG@10
Model                                
BM25   0.280  0.7858  0.6086   0.5886
VSM    0.295  0.6725  0.6320   0.5460
RRF    0.295  0.7609  0.6390   0.5884

--- Stage 2: Real RAGAS Evaluation ---
Generation LLM : GPT-4o-mini (OpenAI)
Judge LLM      : Gemini 2.5 Flash  (Es et al., 2024)
Metrics        : Faithfulness, ResponseRelevancy, ContextUtilization
       faithfulness  answer_relevancy  context_utilization
Model                                                     
BM25         0.9652            0.9084               0.6650
VSM          0.9431            0.9120               0.5910
RRF          0.9735            0.9261               0.6293

Output files saved:
  results_ir_evaluation.csv
  results_ragas_bm25.csv
  results_ragas_vsm.csv
  results_ragas_rrf.csv
  results_ragas_comparison.csv


13. Interactive Query (Final Search)

Ask any question against the QMUL policy corpus.  
Returns the answer, citations, and live RAGAS scores for that single response

In [ ]:
import textwrap

DIVIDER     = '=' * 88
SUB_DIVIDER = '-' * 88


def print_answer_with_citations(result):
    print(DIVIDER)
    print(f"  QUERY  : {result['query']}")
    print(f"  MODEL  : {result['retrieval_model'].upper()}")
    print(DIVIDER)
    print('\n  ANSWER\n')
    wrapped = textwrap.fill(result['answer'], width=70,
                            initial_indent='  ', subsequent_indent='  ')
    print(wrapped)

    if not result['citations']:
        print('\n  (No supporting chunks retrieved.)')
        print(DIVIDER)
        return

    print(f'\n{SUB_DIVIDER}')
    print('  CITATIONS\n')
    for c in result['citations']:
        print(f"  [{c['number']}]  {c['source_title']}")
        print(f"        Chunk : {c['chunk_id']}")
        print(f"        Pages : {c['pages']}")
        print()

    print(SUB_DIVIDER)
    print('  CONTEXT SNIPPETS  (first 300 chars of each retrieved chunk)\n')
    for i, chunk in enumerate(result['chunks_used'], start=1):
        snippet = chunk['text'][:300].replace('\n', ' ')
        print(f'  [{i}]  {snippet}...')
        print()
    print(DIVIDER)


def score_single_query(result):
    if not result['chunks_used']:
        print('  (No chunks retrieved — RAGAS scores not available)')
        return

    print('  RAGAS QUALITY SCORES  (Gemini 2.5 Flash judge)\n')

    single_sample = [{
        'user_input':         result['query'],
        'response':           result['answer'],
        'retrieved_contexts': [c['text'] for c in result['chunks_used']],
    }]

    try:
        single_dataset = EvaluationDataset.from_list(single_sample)
        scores = evaluate(
            dataset=single_dataset,
            metrics=RAGAS_METRICS,
            llm=ragas_llm,
            embeddings=ragas_embeddings,
        ).to_pandas()

        label_map = {
            'faithfulness':       'Faithfulness',
            'response_relevancy': 'Answer Relevance',
            'answer_relevancy':   'Answer Relevance',
            'context_utilization':'Context Utilization',
        }
        for col, label in label_map.items():
            if col in scores.columns:
                val = scores[col].iloc[0]
                try:
                    print(f'    {label:<22} : {float(val):.4f}')
                except (TypeError, ValueError):
                    print(f'    {label:<22} : N/A')

        print()
        print('  Reference: Es et al. (2024) — RAGAS evaluation framework.')

    except Exception as e:
        print(f'  RAGAS scoring failed: {e}')

    print(DIVIDER)


def interactive_query():
    print(DIVIDER)
    print('  QMUL POLICY RAG SYSTEM - Interactive Query')
    print('  Corpus: 20 Finance & Governance policy documents (237 chunks)')
    print(DIVIDER)

    query = input('\n  Enter your question: ').strip()
    if not query:
        print('  No query entered. Please re-run the cell.')
        return

    print()
    print('  Retrieval model options:')
    print('    [1] bm25  - BM25 Okapi (Robertson & Zaragoza, 2009) [default]')
    print('    [2] vsm   - TF-IDF Vector Space Model (Salton & Buckley, 1988)')
    print('    [3] rrf   - Reciprocal Rank Fusion (Cormack et al., 2009)')
    model_choice = input('\n  Select model [1/2/3] (default: 1 - BM25): ').strip()

    model_map = {'1': 'bm25', '2': 'vsm', '3': 'rrf',
                 '': 'bm25', 'bm25': 'bm25', 'vsm': 'vsm', 'rrf': 'rrf'}
    model = model_map.get(model_choice.lower(), 'bm25')

    print(f'\n  Retrieving with {model.upper()} and generating answer...\n')
    result = generate_answer(query, model=model)
    print_answer_with_citations(result)

    print(f'\n  Running RAGAS evaluation on this response...\n')
    score_single_query(result)


interactive_query()

  QMUL POLICY RAG SYSTEM - Interactive Query
  Corpus: 20 Finance & Governance policy documents (237 chunks)

  Enter your question: what is the immediate action by appropriate authority for a fraud response?

  Retrieval model options:
    [1] bm25  - BM25 Okapi (Robertson & Zaragoza, 2009) [default]
    [2] vsm   - TF-IDF Vector Space Model (Salton & Buckley, 1988)
    [3] rrf   - Reciprocal Rank Fusion (Cormack et al., 2009)

  Select model [1/2/3] (default: 1 - BM25): 1

  Retrieving with BM25 and generating answer...

  QUERY  : what is the immediate action by appropriate authority for a fraud response?
  MODEL  : BM25

  ANSWER

  The immediate action for a fraud response is for an employee to
  discuss their suspicions as soon as possible with their Line Manager
  and the Chief Financial Officer. If the Line Manager or Chief
  Financial Officer is the subject of any allegations, the employee
  should go directly to the Chief Financial Officer. If the suspicion
  involves another

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/lib/python3.12/threading.py:287: RuntimeWarning: coroutine 'ClientResponse.json' was never awaited
  if hasattr(lock, '_release_save'):


    Faithfulness           : 0.6667
    Answer Relevance       : 0.7457
    Context Utilization    : 0.3333

  Reference: Es et al. (2024) — RAGAS evaluation framework.


---
## References

Cormack, G.V., Clarke, C.L.A. and Buettcher, S. (2009). Reciprocal rank fusion outperforms Condorcet and individual rank learning methods. *Proceedings of the 32nd ACM SIGIR*, pp. 758–759.

Es, S., James, J., Anke, L.E. and Schockaert, S. (2024). RAGAS: Automated evaluation of retrieval augmented generation. *Proceedings of EACL 2024*, pp. 150–158.

Manning, C.D., Raghavan, P. and Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.

Robertson, S. and Zaragoza, H. (2009). The probabilistic relevance framework: BM25 and beyond. *Foundations and Trends in IR*, 3(4), pp. 333–389.

Rocchio, J.J. (1971). Relevance feedback in information retrieval. In G. Salton (ed.), *The SMART Retrieval System*, pp. 313–323. Prentice-Hall.

Salton, G. and Buckley, C. (1988). Term-weighting approaches in automatic text retrieval. *Information Processing & Management*, 24(5), pp. 513–523.

Sanderson, M. (2013). Test collection based evaluation of IR systems. *Foundations and Trends in IR*, 4(4), pp. 247–375.

Xu, J. and Croft, W.B. (1996). Query expansion using local and global document analysis. *Proceedings of the 19th ACM SIGIR*, pp. 4–11.

Zhai, C. and Lafferty, J. (2001). A study of smoothing methods for language models applied to ad hoc IR. *Proceedings of the 24th ACM SIGIR*, pp. 334–342.